In [12]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import holidays
from sklearn.model_selection import train_test_split # 이 부분은 시계열 분할로 대체됩니다.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# from google.colab import drive # 로컬 환경에서는 이 줄을 주석 처리하거나 삭제하세요.

drive.mount('/content/drive') # 로컬 환경에서는 이 줄을 주석 처리하거나 삭제하세요.
PATH = "/content/drive/MyDrive/열수요예측/" # 로컬 환경에 맞게 경로를 수정하세요.


def preprocess(df, is_train=True, feature_columns=None):
    """
    데이터 전처리 함수
    is_train=True일 경우, 학습 데이터를 전처리하며 target(y)을 생성합니다.
    is_train=False일 경우, 테스트 데이터를 전처리합니다.
    """
    df = df.copy()
    # 컬럼명 소문자 변경 및 정리
    df.columns = df.columns.str.lower()
    df.columns = [col.split('.')[1] if '.' in col else col for col in df.columns]

    # -99.0 값을 결측치로 변환
    df.replace(-99.0, np.nan, inplace=True)

    # tm 변수를 datetime으로 변환
    df['tm'] = pd.to_datetime(df['tm'], format='%Y%m%d%H')

    # 강사님 피드백: 결측치 처리
    # 일사량(si)은 밤 시간에 0이므로 0으로 채우는 것이 합리적일 수 있습니다.
    # rn_hr1(시간당 강수량)도 0으로 채웁니다.
    if 'si' in df.columns:
        df['si'].fillna(0, inplace=True)
    if 'rn_hr1' in df.columns:
        df['rn_hr1'].fillna(0, inplace=True)

    # 풍향(wd)과 풍속(ws) 등 다른 결측치는 지점별 특성을 고려하여 앞 시간 데이터로 채웁니다(ffill).
    # ffill 후에도 남는 결측치(데이터 시작 부분)는 bfill로 채웁니다.
    df = df.sort_values(by=['branch_id', 'tm'])
    df.fillna(method='ffill', inplace=True)
    df.fillna(method='bfill', inplace=True)

    # --- 파생 변수 생성 ---
    # 시간 관련 변수
    df['month'] = df['tm'].dt.month
    df['hour'] = df['tm'].dt.hour
    df['weekday'] = df['tm'].dt.dayofweek
    df['dayofyear'] = df['tm'].dt.dayofyear
    df['is_weekend'] = (df['weekday'] >= 5).astype(int)

    # 시간의 주기성을 나타내는 sin/cos 변환
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

    # 풍향(wd)을 sin/cos 변환
    if 'wd' in df.columns:
        df['wd_rad'] = np.deg2rad(df['wd'])
        df['wd_sin'] = np.sin(df['wd_rad'])
        df['wd_cos'] = np.cos(df['wd_rad'])
        df.drop(['wd', 'wd_rad'], axis=1, inplace=True)

    # 상호작용 변수
    df['ta_hm'] = df['ta'] * df['hm']
    df['ta_ws'] = df['ta'] * df['ws']

    # 강사님 피드백: branch_ID별로 그룹화하여 시계열 특성 생성
    # Lag features (이전 시간대 값)
    if is_train and 'heat_demand' in df.columns:
        # 1시간 전 열수요를 변수로 사용
        df['lag_1hr_heat_demand'] = df.groupby('branch_id')['heat_demand'].shift(1)

    # Rolling features (이동 평균/표준편차)
    for window in [3, 6, 12]:
        df[f'ta_rolling_mean_{window}'] = df.groupby('branch_id')['ta'].transform(lambda x: x.rolling(window).mean())
        df[f'ws_rolling_mean_{window}'] = df.groupby('branch_id')['ws'].transform(lambda x: x.rolling(window).mean())

    # 다시 ffill/bfill로 롤링/랙 변수 생성 시 발생하는 NaN 처리
    df.fillna(method='ffill', inplace=True)
    df.fillna(method='bfill', inplace=True)

    # [수정] 필요 없는 원본 컬럼 제거 시 'tm' 컬럼은 남겨둡니다.
    df.drop(['weekday', 'dayofyear'], axis=1, inplace=True)

    # branch_ID를 원-핫 인코딩
    df = pd.get_dummies(df, columns=['branch_id'], drop_first=True)

    if is_train:
        # --- 1. 시점 문제 해결: target(heat_demand)을 한 시간 뒤의 값으로 설정 ---
        # 현재 행의 feature로 다음 시간의 heat_demand를 예측
        # one-hot-encoding된 branch_id_... 컬럼들을 그룹화 키로 사용합니다.
        branch_cols = [col for col in df.columns if 'branch_id_' in col]
        df['target'] = df.groupby(branch_cols)['heat_demand'].shift(-1)

        # target이 NaN인 마지막 행들은 학습에 사용할 수 없으므로 제거
        df.dropna(subset=['target'], inplace=True)

        # [수정] 여기서는 X, y를 분리하지 않고, 전체 데이터프레임을 반환합니다.
        # feature_columns도 tm을 제외하고 생성합니다.
        if feature_columns is None:
            feature_columns = df.drop(['heat_demand', 'target', 'tm'], axis=1).columns.tolist()
        return df, feature_columns

    else: # is_train=False (테스트 데이터)
        # 테스트 데이터에서는 target을 만들지 않음
        # [수정] 테스트 데이터에서도 'tm' 컬럼은 예측에 사용되지 않으므로 삭제합니다.
        X = df.drop(['heat_demand', 'tm'], axis=1, errors='ignore')
        # 학습 데이터의 컬럼에 맞춰 추가/삭제
        X = X.reindex(columns=feature_columns, fill_value=0)
        return X

def train_and_save_model():
    """
    모델 학습 및 저장 함수
    """
    df_train_raw = pd.read_csv(PATH + "train_heat.csv")

    # [수정] 전처리 함수가 전체 데이터프레임과 피처 컬럼 리스트를 반환합니다.
    processed_df, feature_columns = preprocess(df_train_raw, is_train=True)

    # --- 2. 시계열 데이터 분리 (시간순) ---
    # 강사님 피드백: 랜덤 샘플링 대신 시간순으로 데이터를 분할합니다.
    # 예: 2023년 데이터를 검증 데이터로 사용

    # [수정] 이제 'tm' 컬럼이 있으므로 직접 사용하여 데이터를 분리합니다.
    split_date = pd.to_datetime('2023-01-01')
    train_df = processed_df[processed_df['tm'] < split_date]
    val_df = processed_df[processed_df['tm'] >= split_date]

    # [수정] 분리된 데이터에서 X와 y를 만들고, 'tm' 컬럼을 최종적으로 제거합니다.
    y_train = train_df['target']
    x_train = train_df[feature_columns] # 저장된 피처 컬럼명 사용

    y_val = val_df['target']
    x_val = val_df[feature_columns] # 저장된 피처 컬럼명 사용

    print(f"Train data shape: {x_train.shape}, Validation data shape: {x_val.shape}")

    model = lgb.LGBMRegressor(
        n_estimators=10000,
        learning_rate=0.01,
        num_leaves=128,
        max_depth=16,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    print("🚀 Model training started...")
    model.fit(
        x_train, y_train,
        eval_set=[(x_val, y_val)],
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
    )

    preds = model.predict(x_val)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae = mean_absolute_error(y_val, preds)
    r2 = r2_score(y_val, preds)

    print("\n✅ Validation Performance (Time-Series Split):")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"R2  : {r2:.4f}")

    joblib.dump(model, PATH + 'lgb_model_v2.pkl')
    joblib.dump(feature_columns, PATH + 'feature_columns_v2.pkl')
    print("\n💾 Model and feature columns saved successfully.")

    return feature_columns


def predict_and_submit(test_file_path, output_csv_name, feature_columns):
    """
    테스트 데이터 예측 및 제출 파일 생성 함수
    """
    df_test_raw = pd.read_csv(test_file_path) # 원본 테스트 파일 읽기

    # 학습 시 사용된 컬럼 정보를 가지고 테스트 데이터 전처리
    df_test_proc = preprocess(df_test_raw.copy(), is_train=False, feature_columns=feature_columns) # [수정] 원본 보존을 위해 copy() 사용

    model = joblib.load(PATH + 'lgb_model_v2.pkl')

    predictions = model.predict(df_test_proc)

    predictions[predictions < 0] = 0

    # [수정] submission 파일을 읽는 대신, 원본 테스트 데이터프레임에 예측 결과를 추가합니다.
    df_test_raw['heat_demand'] = predictions

    # [수정] 결과를 저장합니다.
    df_test_raw.to_csv(f"{PATH}{output_csv_name}.csv", index=False)
    print(f"📁 Submission file saved: {output_csv_name}.csv")

# --- 코드 실행 ---
if __name__ == '__main__':
    # 1. 모델 학습 및 저장
    trained_feature_columns = train_and_save_model()

    # 2. 테스트 데이터 예측 및 제출
    predict_and_submit(
        test_file_path=PATH + "test_heat.csv",
        output_csv_name="submission_final.csv",
        feature_columns=trained_feature_columns
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipython-input-12-3325837114.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['si'].fillna(0, inplace=True)
/tmp/ipython-input-12-3325837114.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.

Train data shape: (332861, 45), Validation data shape: (166421, 45)
🚀 Model training started...
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 42.8126	valid_0's l2: 1832.92
[200]	valid_0's rmse: 20.1915	valid_0's l2: 407.695
[300]	valid_0's rmse: 14.0922	valid_0's l2: 198.589
[400]	valid_0's rmse: 12.6974	valid_0's l2: 161.225
[500]	valid_0's rmse: 12.3058	valid_0's l2: 151.433
[600]	valid_0's rmse: 12.1029	valid_0's l2: 146.481
[700]	valid_0's rmse: 11.9754	valid_0's l2: 143.41
[800]	valid_0's rmse: 11.8601	valid_0's l2: 140.662
[900]	valid_0's rmse: 11.7877	valid_0's l2: 138.949
[1000]	valid_0's rmse: 11.7449	valid_0's l2: 137.943
[1100]	valid_0's rmse: 11.7179	valid_0's l2: 137.308
[1200]	valid_0's rmse: 11.6991	valid_0's l2: 136.87
[1300]	valid_0's rmse: 11.6807	valid_0's l2: 136.439
[1400]	valid_0's rmse: 11.6695	valid_0's l2: 136.177
[1500]	valid_0's rmse: 11.6594	valid_0's l2: 135.942
[1600]	valid_0's rmse: 11.6515	valid_0's l2: 135.757
[1700

/tmp/ipython-input-12-3325837114.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['si'].fillna(0, inplace=True)
/tmp/ipython-input-12-3325837114.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.

📁 Submission file saved: submission_final.csv.csv
